In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [ ]:
import transformers
import datasets
import evaluate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Transformers: 5.15.1
Datasets: 4.0.0


In [ ]:
import pandas as pd
import os

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())
print("\nSample:")
display(train_df.head())

Train: (19824, 2)
Validation: (4957, 2)
Test: (25000, 2)

Columns: ['text', 'label']

Sample:


,text,label
0,This is why I still have nightmares.<br /><br ...,1
1,For those who never saw A CHORUS LINE onstage ...,0
2,This is a romantic comedy where Albert Einstei...,1
3,"In April of 1965, CBS broadcast the first of B...",1
4,Loony Tunes have ventured (at least) twice int...,0


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
SMOKE_TRAIN_SIZE = 2000
SMOKE_VAL_SIZE = 500

smoke_train_df = train_df.sample(
    n=SMOKE_TRAIN_SIZE,
    random_state=42,
    stratify=train_df["label"]
)

smoke_val_df = val_df.sample(
    n=SMOKE_VAL_SIZE,
    random_state=42,
    stratify=val_df["label"]
)

print("Smoke train:", smoke_train_df.shape)
print("Smoke validation:", smoke_val_df.shape)

print("\nTrain label distribution:")
print(smoke_train_df["label"].value_counts())

print("\nValidation label distribution:")
print(smoke_val_df["label"].value_counts())

TypeError: NDFrame.sample() got an unexpected keyword argument 'stratify'

In [ ]:
from sklearn.model_selection import train_test_split

SMOKE_TRAIN_SIZE = 2000
SMOKE_VAL_SIZE = 500

# Stratified sample for smoke-test training data
smoke_train_df, _ = train_test_split(
    train_df,
    train_size=SMOKE_TRAIN_SIZE,
    random_state=42,
    stratify=train_df["label"]
)

# Stratified sample for smoke-test validation data
smoke_val_df, _ = train_test_split(
    val_df,
    train_size=SMOKE_VAL_SIZE,
    random_state=42,
    stratify=val_df["label"]
)

print("Smoke train:", smoke_train_df.shape)
print("Smoke validation:", smoke_val_df.shape)

print("\nTrain label distribution:")
print(smoke_train_df["label"].value_counts())

print("\nValidation label distribution:")
print(smoke_val_df["label"].value_counts())

Smoke train: (2000, 2)
Smoke validation: (500, 2)

Train label distribution:
label
1    1004
0     996
Name: count, dtype: int64

Validation label distribution:
label
1    251
0    249
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [ ]:
from datasets import Dataset

def tokenize_data(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

smoke_train_dataset = Dataset.from_pandas(
    smoke_train_df[["text", "label"]],
    preserve_index=False
)

smoke_val_dataset = Dataset.from_pandas(
    smoke_val_df[["text", "label"]],
    preserve_index=False
)

smoke_train_dataset = smoke_train_dataset.map(
    tokenize_data,
    batched=True
)

smoke_val_dataset = smoke_val_dataset.map(
    tokenize_data,
    batched=True
)

print(smoke_train_dataset)
print(smoke_val_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("BERT model loaded successfully.")
print("Model device:", next(model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded successfully.
Model device: cuda:0


In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["precision"],

        "recall": recall_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["recall"],

        "f1": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["f1"]
    }
}

SyntaxError: unmatched '}' (3306692617.py, line 38)

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["precision"],

        "recall": recall_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["recall"],

        "f1": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["f1"]
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_smoke_test",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=50,

    report_to="none",

    fp16=True
)

print("Training configuration ready.")

Training configuration ready.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=smoke_train_dataset,
    eval_dataset=smoke_val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("Trainer ready.")

Trainer ready.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.354851,0.230289,0.926000,0.902256,0.956175,0.928433


TrainOutput(global_step=250, training_loss=0.4238645782470703, metrics={'train_runtime': 33.6279, 'train_samples_per_second': 59.474, 'train_steps_per_second': 7.434, 'total_flos': 263111055360000.0, 'train_loss': 0.4238645782470703, 'epoch': 1.0})

In [ ]:
smoke_results = trainer.evaluate()

print("\nBERT Smoke-Test Results")
print("=" * 40)

for key, value in smoke_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.354851,0.230289,1,0.926000,0.902256,0.956175,0.928433



BERT Smoke-Test Results
eval_loss: 0.2303
eval_accuracy: 0.9260
eval_precision: 0.9023
eval_recall: 0.9562
eval_f1: 0.9284


In [ ]:
import gc
import torch

del trainer
del model

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
full_train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

full_val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

full_train_dataset = full_train_dataset.map(
    tokenize_data,
    batched=True
)

full_val_dataset = full_val_dataset.map(
    tokenize_data,
    batched=True
)

print("Full training dataset:", full_train_dataset)
print("Full validation dataset:", full_val_dataset)

Map:   0%|          | 0/19824 [00:00<?, ? examples/s]

Map:   0%|          | 0/4957 [00:00<?, ? examples/s]

Full training dataset: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 19824
})
Full validation dataset: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4957
})


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("Full BERT model ready.")
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Full BERT model ready.
Device: cuda:0


In [ ]:
full_training_args = TrainingArguments(
    output_dir="./bert_final",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=200,

    report_to="none",

    fp16=True
)

In [ ]:
full_bert_trainer = Trainer(
    model=model,
    args=full_training_args,

    train_dataset=full_train_dataset,
    eval_dataset=full_val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("Full BERT Trainer ready.")

Full BERT Trainer ready.


In [ ]:
bert_train_output = full_bert_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.298010,0.238306,0.924955,0.923569,0.927280,0.925421


In [ ]:
bert_val_results = full_bert_trainer.evaluate()

print("\n" + "=" * 50)
print("BERT — FULL VALIDATION RESULTS")
print("=" * 50)

for key, value in bert_val_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.298010,0.238306,1,0.924955,0.923569,0.927280,0.925421



BERT — FULL VALIDATION RESULTS
eval_loss: 0.2383
eval_accuracy: 0.9250
eval_precision: 0.9236
eval_recall: 0.9273
eval_f1: 0.9254


In [ ]:
bert_result = {
    "Model": "BERT",
    "Accuracy": bert_val_results["eval_accuracy"],
    "Precision": bert_val_results["eval_precision"],
    "Recall": bert_val_results["eval_recall"],
    "F1": bert_val_results["eval_f1"]
}

print(bert_result)

{'Model': 'BERT', 'Accuracy': 0.9249546096429292, 'Precision': 0.9235694277711084, 'Recall': 0.9272800321414223, 'F1': 0.92542101042502}


In [ ]:
import json

with open("bert_validation_results.json", "w") as f:
    json.dump(bert_result, f, indent=4)

print("BERT validation results saved.")

BERT validation results saved.


In [ ]:
import gc
import torch

del full_bert_trainer
del model

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
from transformers import AutoTokenizer

ROBERTA_MODEL_NAME = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL_NAME
)

print("RoBERTa tokenizer loaded.")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

RoBERTa tokenizer loaded.


In [ ]:
def tokenize_roberta(batch):
    return roberta_tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

roberta_train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

roberta_val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

roberta_train_dataset = roberta_train_dataset.map(
    tokenize_roberta,
    batched=True
)

roberta_val_dataset = roberta_val_dataset.map(
    tokenize_roberta,
    batched=True
)

print(roberta_train_dataset)
print(roberta_val_dataset)

Map:   0%|          | 0/19824 [00:00<?, ? examples/s]

Map:   0%|          | 0/4957 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 19824
})
Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 4957
})


In [ ]:
from transformers import AutoModelForSequenceClassification

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL_NAME,
    num_labels=2
)

roberta_model.to(device)

print("RoBERTa model loaded.")
print("Device:", next(roberta_model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded.
Device: cuda:0


In [ ]:
roberta_training_args = TrainingArguments(
    output_dir="./roberta_final",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=200,

    report_to="none",

    fp16=True
)

In [ ]:
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,

    train_dataset=roberta_train_dataset,
    eval_dataset=roberta_val_dataset,

    processing_class=roberta_tokenizer,

    compute_metrics=compute_metrics
)

print("RoBERTa Trainer ready.")

RoBERTa Trainer ready.


In [ ]:
roberta_train_output = roberta_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.287098,0.283400,0.931410,0.924536,0.940137,0.932271


In [ ]:
roberta_val_results = roberta_trainer.evaluate()

print("\n" + "=" * 50)
print("RoBERTa — FULL VALIDATION RESULTS")
print("=" * 50)

for key, value in roberta_val_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.287098,0.283400,1,0.931410,0.924536,0.940137,0.932271



RoBERTa — FULL VALIDATION RESULTS
eval_loss: 0.2834
eval_accuracy: 0.9314
eval_precision: 0.9245
eval_recall: 0.9401
eval_f1: 0.9323


In [ ]:
# ==========================================
# FINAL TEST SET — RoBERTa
# ==========================================

roberta_test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

roberta_test_dataset = roberta_test_dataset.map(
    tokenize_roberta,
    batched=True
)

print(roberta_test_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 25000
})


In [ ]:
roberta_test_results = roberta_trainer.evaluate(
    eval_dataset=roberta_test_dataset
)

print("\n" + "=" * 55)
print("RoBERTa — FINAL TEST RESULTS")
print("=" * 55)

for key, value in roberta_test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.287098,0.276362,1,0.932320,0.923777,0.942400,0.932995



RoBERTa — FINAL TEST RESULTS
eval_loss: 0.2764
eval_accuracy: 0.9323
eval_precision: 0.9238
eval_recall: 0.9424
eval_f1: 0.9330


In [ ]:
roberta_final_result = {
    "Model": "RoBERTa",
    "Accuracy": roberta_test_results["eval_accuracy"],
    "Precision": roberta_test_results["eval_precision"],
    "Recall": roberta_test_results["eval_recall"],
    "F1": roberta_test_results["eval_f1"]
}

print(roberta_final_result)

{'Model': 'RoBERTa', 'Accuracy': 0.93232, 'Precision': 0.9237766624843162, 'Recall': 0.9424, 'F1': 0.9329954063044511}


In [ ]:
print("BERT trainer exists:", "full_bert_trainer" in globals())
print("BERT model exists:", "model" in globals())

BERT trainer exists: False
BERT model exists: False


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

bert_model.to(device)

print("Fresh BERT loaded.")
print("Device:", next(bert_model.parameters()).device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh BERT loaded.
Device: cuda:0


In [2]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
bert_model.to(device)

print("BERT model loaded successfully.")
print("Model device:", next(bert_model.parameters()).device)

BERT model loaded successfully.
Model device: cuda:0


In [6]:
from transformers import TrainingArguments, Trainer

bert_training_args = TrainingArguments(
    output_dir="./bert_final",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=200,

    report_to="none",

    fp16=True
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,

    train_dataset=full_train_dataset,
    eval_dataset=full_val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("BERT Trainer ready.")

NameError: name 'full_train_dataset' is not defined

In [7]:
print("train_df:", "train_df" in globals())
print("val_df:", "val_df" in globals())
print("test_df:", "test_df" in globals())
print("tokenizer:", "tokenizer" in globals())
print("bert_model:", "bert_model" in globals())

train_df: False
val_df: False
test_df: False
tokenizer: False
bert_model: True


In [10]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (19824, 2)
Validation: (4957, 2)
Test: (25000, 2)


In [11]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [12]:
from datasets import Dataset

full_train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

full_val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

print("Train:", full_train_dataset)
print("Validation:", full_val_dataset)

Train: Dataset({
    features: ['text', 'label'],
    num_rows: 19824
})
Validation: Dataset({
    features: ['text', 'label'],
    num_rows: 4957
})


In [13]:
def tokenize_data(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

full_train_dataset = full_train_dataset.map(
    tokenize_data,
    batched=True
)

full_val_dataset = full_val_dataset.map(
    tokenize_data,
    batched=True
)

print("Tokenization complete.")
print(full_train_dataset)
print(full_val_dataset)

Map:   0%|          | 0/19824 [00:00<?, ? examples/s]

Map:   0%|          | 0/4957 [00:00<?, ? examples/s]

Tokenization complete.
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 19824
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4957
})


In [14]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

bert_model.to(device)

print("Device:", device)
print("BERT device:", next(bert_model.parameters()).device)

Device: cuda
BERT device: cuda:0


In [15]:
from transformers import TrainingArguments, Trainer

bert_training_args = TrainingArguments(
    output_dir="./bert_final",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=200,

    report_to="none",

    fp16=True
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,

    train_dataset=full_train_dataset,
    eval_dataset=full_val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("BERT Trainer ready.")

NameError: name 'compute_metrics' is not defined

In [16]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["precision"],

        "recall": recall_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["recall"],

        "f1": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="binary"
        )["f1"]
    }

print("Metrics function ready.")

ModuleNotFoundError: No module named 'evaluate'

In [17]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

print("Metrics function ready.")

Metrics function ready.


In [18]:
from transformers import TrainingArguments, Trainer

bert_training_args = TrainingArguments(
    output_dir="./bert_final",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=200,

    report_to="none",

    fp16=True
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,

    train_dataset=full_train_dataset,
    eval_dataset=full_val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("BERT Trainer ready.")

BERT Trainer ready.


In [19]:
bert_train_output = bert_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.292755,0.237511,0.921929,0.919059,0.926075,0.922554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
bert_val_results = bert_trainer.evaluate()

print("\n" + "=" * 55)
print("BERT — VALIDATION RESULTS")
print("=" * 55)

for key, value in bert_val_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.292755,0.237511,1,0.921929,0.919059,0.926075,0.922554



BERT — VALIDATION RESULTS
eval_loss: 0.2375
eval_accuracy: 0.9219
eval_precision: 0.9191
eval_recall: 0.9261
eval_f1: 0.9226


In [21]:
bert_test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

bert_test_dataset = bert_test_dataset.map(
    tokenize_data,
    batched=True
)

print(bert_test_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})


In [22]:
bert_test_results = bert_trainer.evaluate(
    eval_dataset=bert_test_dataset
)

print("\n" + "=" * 55)
print("BERT — FINAL TEST RESULTS")
print("=" * 55)

for key, value in bert_test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.292755,0.246967,1,0.917720,0.911563,0.925200,0.918331



BERT — FINAL TEST RESULTS
eval_loss: 0.2470
eval_accuracy: 0.9177
eval_precision: 0.9116
eval_recall: 0.9252
eval_f1: 0.9183


In [2]:
import numpy as np
import pandas as pd
from datasets import Dataset

# -------------------------------
# BERT TEST DATASET
# -------------------------------

bert_test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

bert_test_dataset = bert_test_dataset.map(
    tokenize_data,
    batched=True
)

# -------------------------------
# BERT PREDICTIONS
# -------------------------------

bert_output = bert_trainer.predict(
    bert_test_dataset
)

bert_predictions = np.argmax(
    bert_output.predictions,
    axis=-1
)

# -------------------------------
# SAVE
# -------------------------------

bert_pred_df = pd.DataFrame({
    "label": test_df["label"].values,
    "prediction": bert_predictions
})

bert_pred_df.to_csv(
    "/content/bert_test_predictions.csv",
    index=False
)

print("BERT predictions saved.")
print(bert_pred_df.head())
print("Shape:", bert_pred_df.shape)

NameError: name 'test_df' is not defined

In [3]:
print("test_df:", "test_df" in globals())
print("bert_trainer:", "bert_trainer" in globals())
print("roberta_trainer:", "roberta_trainer" in globals())

test_df: False
bert_trainer: False
roberta_trainer: False


In [4]:
import os

print("Files in /content:")
for item in os.listdir("/content"):
    print(item)

Files in /content:
.config
validation.csv
test.csv
train.csv
sample_data


In [5]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if (
            "checkpoint" in root.lower()
            or "bert" in file.lower()
            or "roberta" in file.lower()
            or file.endswith(".csv")
        ):
            print(os.path.join(root, file))

/content/validation.csv
/content/test.csv
/content/train.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv


In [6]:
print("test.csv exists:", os.path.exists("/content/test.csv"))
print("train.csv exists:", os.path.exists("/content/train.csv"))
print("validation.csv exists:", os.path.exists("/content/validation.csv"))

test.csv exists: True
train.csv exists: True
validation.csv exists: True
